In [7]:
import pandas as pd
import numpy as np
import tensorflow as tf

train=pd.read_csv("engtamilTrain.csv")
train=train.drop(["Unnamed: 0"],axis=1)
english_sentences=train["en"]
tamil_sentence=train['ta']
english_sentences=english_sentences.head(1000)
tamil_sentences=tamil_sentence.head(1000)

In [8]:
#Function to add SOS and EOS to the statement

train=pd.read_csv("engtamilTrain.csv")
train=train.drop(["Unnamed: 0"],axis=1)
english_sentences=train["en"]
tamil_sentence=train['ta']
english_sentences=english_sentences.head(1000)
tamil_sentences=tamil_sentence.head(1000)

def addSosEos(seriesSentence):
    # Define the <SOS> and <EOS> tokens
    sos_token = "<SOS>"
    eos_token = "<EOS>"

    # Add <SOS> and <EOS> tokens to each statement
    statements_with_tokens = [f"{sos_token} {statement} {eos_token}" for statement in seriesSentence]

    english_sent=[]
    # Print the statements with tokens
    for statement in statements_with_tokens:
        english_sent.append(statement)
      #  print(statement)
    return english_sent

In [9]:
english_sent_SE=addSosEos(english_sentences)
tamil_sent_SE=addSosEos(tamil_sentences)

In [10]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [11]:
# Tokenize the English and Tamil sentences
english_tokenizer = Tokenizer(filters="")
english_tokenizer.fit_on_texts(english_sent_SE)
english_vocab_size = len(english_tokenizer.word_index) + 1
english_sequences = english_tokenizer.texts_to_sequences(english_sent_SE)

In [12]:
tamil_tokenizer = Tokenizer(filters="")
tamil_tokenizer.fit_on_texts(tamil_sent_SE)
tamil_vocab_size = len(tamil_tokenizer.word_index) + 1
tamil_sequences = tamil_tokenizer.texts_to_sequences(tamil_sent_SE)

In [13]:
max_input_seq_length=20
max_output_seq_length=20

In [14]:
# Pad sequences to a fixed length
input_sequences = pad_sequences(english_sequences, maxlen=max_input_seq_length, padding='post')
output_sequences = pad_sequences(tamil_sequences, maxlen=max_output_seq_length, padding='post')

In [15]:
# Prepare the decoder input and output sequences for teacher forcing
decoder_input_sequences = np.zeros_like(output_sequences)
decoder_input_sequences[:, 1:] = output_sequences[:, :-1]
decoder_input_sequences[:, 0] = tamil_tokenizer.word_index['<sos>']

decoder_output_sequences = np.eye(tamil_vocab_size)[output_sequences]

In [18]:
from gensim.models import Word2Vec

eng_model = Word2Vec.load('engmodel.bin')
tam_model = Word2Vec.load('tammodel.bin')

In [19]:
def create_embedding_matrix(word2vec_model, tokenizer, vocab_size):
    embedding_matrix = np.zeros((vocab_size, word2vec_model.vector_size))
    for word, i in tokenizer.word_index.items():
        try:
            embedding_vector = word2vec_model.wv[word]
            embedding_matrix[i] = embedding_vector
        except KeyError:
            pass  # Words not found in the embedding index will be all zeros
    return embedding_matrix

eng_embedding_matrix = create_embedding_matrix(eng_model, english_tokenizer, english_vocab_size)
tam_embedding_matrix = create_embedding_matrix(tam_model, tamil_tokenizer, tamil_vocab_size)

In [20]:
eng_embedding_matrix.shape

(6979, 100)

In [21]:
tam_embedding_matrix.shape

(9922, 100)

In [22]:
def create_seq2seq_model(input_vocab_size, output_vocab_size, input_seq_length, output_seq_length, hidden_units, eng_embedding_matrix, tam_embedding_matrix):
    # Encoder
    encoder_inputs = Input(shape=(input_seq_length,))
    encoder_embedding = Embedding(input_vocab_size, hidden_units, weights=[eng_embedding_matrix], trainable=False)(encoder_inputs)
    encoder_lstm, encoder_state_h, encoder_state_c = LSTM(hidden_units, return_state=True)(encoder_embedding)

    # Decoder
    decoder_inputs = Input(shape=(output_seq_length,))
    decoder_embedding = Embedding(output_vocab_size, hidden_units, weights=[tam_embedding_matrix], trainable=False)(decoder_inputs)
    decoder_lstm = LSTM(hidden_units, return_sequences=True, return_state=True)
    decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=[encoder_state_h, encoder_state_c])
    decoder_dense = Dense(output_vocab_size, activation='softmax')
    decoder_outputs = decoder_dense(decoder_outputs)

    model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
    return model

In [23]:
# Convert target_sequences to one-hot encoded format
target_sequences = tf.keras.utils.to_categorical(output_sequences, num_classes=tamil_vocab_size)

In [24]:
model = create_seq2seq_model(english_vocab_size, tamil_vocab_size, max_input_seq_length, max_output_seq_length, 100, eng_embedding_matrix, tam_embedding_matrix)

In [25]:
# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [26]:
# Fit the model to the data
batch_size = 32
epochs = 500
model.fit([input_sequences, output_sequences], decoder_output_sequences, batch_size=batch_size, epochs=epochs, validation_split=0.2)

Epoch 1/500
25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 69ms/step - accuracy: 0.2466 - loss: 9.0085 - val_accuracy: 0.2598 - val_loss: 8.2237
Epoch 2/500
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 57ms/step - accuracy: 0.2682 - loss: 7.3627 - val_accuracy: 0.2637 - val_loss: 7.3301
Epoch 3/500
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 57ms/step - accuracy: 0.2664 - loss: 6.5751 - val_accuracy: 0.2718 - val_loss: 7.5147
Epoch 4/500
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 56ms/step - accuracy: 0.2682 - loss: 6.4643 - val_accuracy: 0.2700 - val_loss: 7.6316
Epoch 5/500
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 55ms/step - accuracy: 0.2674 - loss: 6.4003 - val_accuracy: 0.2700 - val_loss: 7.7026
Epoch 6/500
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 54ms/step - accuracy: 0.2678 - loss: 6.3517 - val_accuracy: 0.2700 - val_loss: 7.7337
Epoch 7/500
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 54ms/step - accuracy: 0.2678 - loss: 6.3120 - val_accuracy: 0.2700 - val_loss: 7.8153
Epoch 8/500
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 54ms/step - accuracy: 0.2678 - loss: 6.2803 - val_accuracy: 0.

In [27]:
# Preprocessing the input
input_sentence = "<sos>Finally, the columnist fails to tell us who among the political leaders of the bourgeoisie, past and present, he counts among the paragons of morality<eos>"

# Convert the input sentence to sequence
input_sequence = english_tokenizer.texts_to_sequences([input_sentence])

# Pad the statement to the maximum input sequence length
input_sequence = pad_sequences(input_sequence, maxlen=max_input_seq_length, padding='post')

# Generate predictions
predictions = model.predict([input_sequence, np.zeros((1, max_output_seq_length))])

# Convert predictions to tokens
predicted_tokens = np.argmax(predictions, axis=-1)[0]

# Create index to word mapping for Tamil vocabulary
tamil_index_word = {i: w for w, i in tamil_tokenizer.word_index.items()}


# Convert tokens to text
decoded_sentence = []
for token in predicted_tokens:
    if token == 0:  # Assuming 0 is the padding token
        continue
    word = tamil_index_word.get(token)
    if word == '<eos>':
        break
    if word is not None:
        decoded_sentence.append(word)
    else:
        decoded_sentence.append('<unk>')

# Join the words to form the decoded statement
decoded_statement = ' '.join(decoded_sentence)

# Print the decoded statement
print(decoded_statement)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 175ms/step
<sos> கட்சி) pds- திரும்பியுள்ள நீ பேரைக்கொண்ட முதலாவது வலியுறுத்தும் திருப்பிப் நாட்டையும், தனிமைப்படுத்தி, இழிவுபடுத்த


In [28]:
predictions

array([[[1.18838607e-05, 2.64662653e-01, 2.38432699e-06, ...,
         1.93733515e-10, 1.52105814e-10, 1.89945087e-10],
        [6.52459812e-06, 1.61021322e-04, 5.68073210e-06, ...,
         2.70339329e-09, 2.43162646e-09, 2.84938939e-09],
        [5.27452585e-06, 2.72647117e-06, 1.30324333e-05, ...,
         5.00480679e-09, 4.94935826e-09, 5.40809220e-09],
        ...,
        [2.03710347e-01, 3.94657291e-05, 7.19992146e-02, ...,
         4.37973019e-10, 3.56995822e-10, 3.96381983e-10],
        [2.39324301e-01, 1.92928856e-05, 1.22941725e-01, ...,
         1.75212206e-10, 1.57452412e-10, 1.59314506e-10],
        [5.03661931e-01, 1.64441630e-07, 4.76903468e-01, ...,
         9.76950899e-12, 8.59765037e-12, 8.86336577e-12]]],
      shape=(1, 20, 9922), dtype=float32)

In [29]:
predicted_tokens

array([   1, 3707, 3708, 4288,   87, 4290,  638, 4393, 1042, 4394, 4395,
       4396,    0,    0,    0,    0,    0,    0,    0,    0])

In [30]:
input_sequence

array([[   6,  569,   35,   41,  172,    1,   67,  452,    4,    1, 1118,
         226,    5, 1119,   16, 2057,  172,    1, 2058,    4]],
      dtype=int32)